In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# 4-class 5-fold cross-validation experiment

import json
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

SEED = 42

TARGET_INTENTS = [
    "Reveal",
    "Interpret",
    "Revise",
    "Chitchat"
]

N_FOLDS = 5 #5cv expt

random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)
print("Target intents:", TARGET_INTENTS)
print("Number of folds:", N_FOLDS)

Seed: 42
Target intents: ['Reveal', 'Interpret', 'Revise', 'Chitchat']
Number of folds: 5


In [9]:
DATA_PATH = "/content/drive/MyDrive/conversation_en.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("Raw conversations:", len(raw_data))

Raw conversations: 826


In [10]:
conv_ids = [conv["conv_id"] for conv in raw_data]

print("Total conversations:", len(conv_ids))
print("Unique conversation IDs:", len(set(conv_ids)))

duplicate_ids = [
    conv_id
    for conv_id, count in Counter(conv_ids).items()
    if count > 1
]

print("Duplicate conversation IDs:", duplicate_ids)

Total conversations: 826
Unique conversation IDs: 826
Duplicate conversation IDs: []


In [11]:
conversation_summary = []

for conv in raw_data:
    conv_id = conv["conv_id"]

    counts = {
        intent: 0
        for intent in TARGET_INTENTS
    }

    inquire_count = 0

    for msg in conv["conversation"]:

        # We classify user turns only
        if msg.get("role") != "user":
            continue

        intent = msg.get("intent")

        if intent in TARGET_INTENTS:
            counts[intent] += 1

        elif intent == "Inquire":
            inquire_count += 1

    total_4class = sum(counts.values())

    conversation_summary.append({
        "conv_id": conv_id,
        "Reveal": counts["Reveal"],
        "Interpret": counts["Interpret"],
        "Revise": counts["Revise"],
        "Chitchat": counts["Chitchat"],
        "Inquire": inquire_count,
        "total_4class": total_4class
    })

audit_df = pd.DataFrame(conversation_summary)

print("Conversations in audit table:", len(audit_df))

display(audit_df.head(10))

Conversations in audit table: 826


,conv_id,Reveal,Interpret,Revise,Chitchat,Inquire,total_4class
0,19762,1,7,2,0,0,10
1,20322,1,4,0,1,0,6
2,20534,1,6,0,0,0,7
3,20771,1,7,0,1,0,9
4,21152,1,1,0,1,0,3
5,21443,1,6,0,0,0,7
6,21951,1,2,0,0,0,3
7,22766,7,0,0,0,0,7
8,23826,1,1,0,1,1,3
9,24741,1,1,0,1,0,3


In [12]:
eligible_df = audit_df[
    audit_df["total_4class"] > 0
].copy()

eligible_df = eligible_df.reset_index(drop=True)

print("Eligible 4-class conversations:", len(eligible_df))

Eligible 4-class conversations: 825


In [13]:
excluded_df = audit_df[
    audit_df["total_4class"] == 0
].copy()

print(
    "Conversations with zero 4-class target turns:",
    len(excluded_df)
)

display(excluded_df)

Conversations with zero 4-class target turns: 1


,conv_id,Reveal,Interpret,Revise,Chitchat,Inquire,total_4class
302,64952,0,0,0,0,2,0


In [14]:
# Number of eligible conversations comes from the actual filtered data
N_CONVERSATIONS = len(eligible_df)

# Check that equal-sized folds are possible
assert N_CONVERSATIONS % N_FOLDS == 0, (
    f"{N_CONVERSATIONS} conversations cannot be divided "
    f"equally into {N_FOLDS} folds."
)

# Calculate fold size directly from the data
CONVERSATIONS_PER_FOLD = N_CONVERSATIONS // N_FOLDS

print("Eligible conversations:", N_CONVERSATIONS)
print("Number of folds:", N_FOLDS)
print("Conversations per fold:", CONVERSATIONS_PER_FOLD)

Eligible conversations: 825
Number of folds: 5
Conversations per fold: 165


In [15]:
# Verify total class counts before any shuffle/split

class_totals = eligible_df[TARGET_INTENTS].sum()

print("4-class totals:")
print(class_totals)

print("\nTotal labeled user turns:", class_totals.sum())

4-class totals:
Reveal       1031
Interpret    1813
Revise         67
Chitchat      268
dtype: int64

Total labeled user turns: 3179


In [16]:
# Shuffle the 825 eligible conversations ONCE

shuffled_df = eligible_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("Number of shuffled conversations:", len(shuffled_df))

print("\nFirst 20 shuffled conversation IDs:")
print(shuffled_df["conv_id"].head(20).tolist())

Number of shuffled conversations: 825

First 20 shuffled conversation IDs:
[104531, 7487, 37857, 6508, 61169, 68988, 129429, 42246, 94924, 117490, 81418, 59854, 78208, 28400, 74565, 111226, 31061, 14412, 126381, 72725]


In [17]:
# Start fresh from the ONE-TIME shuffled_df

folds = {
    fold_id: []
    for fold_id in range(1, N_FOLDS + 1)
}

fold_class_counts = {
    fold_id: {
        intent: 0
        for intent in TARGET_INTENTS
    }
    for fold_id in range(1, N_FOLDS + 1)
}

In [18]:
# How many complete conversations contain each intent?

print("Number of conversations containing each intent:\n")

for intent in TARGET_INTENTS:
    n_conversations = (shuffled_df[intent] > 0).sum()

    print(
        f"{intent}: "
        f"{n_conversations} conversations"
    )

Number of conversations containing each intent:

Reveal: 778 conversations
Interpret: 665 conversations
Revise: 53 conversations
Chitchat: 212 conversations


In [19]:
# Distribution of number of target turns per conversation

print("\nTarget-turn count per conversation:")
print(
    shuffled_df["total_4class"]
    .value_counts()
    .sort_index()
)


Target-turn count per conversation:
total_4class
1      18
2      98
3     222
4     208
5     227
6      39
7       9
8       2
9       1
10      1
Name: count, dtype: int64


In [20]:
# Add original shuffled position so we preserve the one-time shuffle
work_df = shuffled_df.copy()
work_df["shuffle_order"] = range(len(work_df))

# Boolean presence of each class in each conversation
for intent in TARGET_INTENTS:
    work_df[f"has_{intent}"] = work_df[intent] > 0

display(
    work_df[
        [
            "conv_id",
            "Reveal",
            "Interpret",
            "Revise",
            "Chitchat",
            "total_4class",
            "shuffle_order"
        ]
    ].head()
)

,conv_id,Reveal,Interpret,Revise,Chitchat,total_4class,shuffle_order
0,104531,1,1,0,0,2,0
1,7487,1,3,0,0,4,1
2,37857,1,4,0,0,5,2
3,6508,1,4,0,0,5,3
4,61169,1,0,1,0,2,4


In [21]:
folds = {
    fold_id: []
    for fold_id in range(1, N_FOLDS + 1)
}

assigned_ids = set()

In [22]:
def assign_round_robin(rows):
    """
    Assign COMPLETE conversations one at a time,
    always preferring the currently smallest fold.
    """

    for _, row in rows.iterrows():

        conv_id = int(row["conv_id"])

        if conv_id in assigned_ids:
            continue

        # folds with room remaining
        available_folds = [
            f for f in range(1, N_FOLDS + 1)
            if len(folds[f]) < CONVERSATIONS_PER_FOLD
        ]

        # choose smallest fold;
        # fold number resolves exact ties deterministically
        chosen_fold = min(
            available_folds,
            key=lambda f: (len(folds[f]), f)
        )

        folds[chosen_fold].append(conv_id)
        assigned_ids.add(conv_id)

In [23]:
revise_df = (
    work_df[work_df["Revise"] > 0]
    .sort_values(
        ["Revise", "shuffle_order"],
        ascending=[False, True],
        kind="stable"
    )
)

print("Revise-containing conversations:", len(revise_df))

assign_round_robin(revise_df)

Revise-containing conversations: 53


In [24]:
for f in range(1, N_FOLDS + 1):

    subset = work_df[
        work_df["conv_id"].isin(folds[f])
    ]

    print(
        f"Fold {f}: "
        f"{len(folds[f])} conversations, "
        f"Revise turns = {subset['Revise'].sum()}, "
        f"Revise-containing conversations = "
        f"{(subset['Revise'] > 0).sum()}"
    )

Fold 1: 11 conversations, Revise turns = 15, Revise-containing conversations = 11
Fold 2: 11 conversations, Revise turns = 14, Revise-containing conversations = 11
Fold 3: 11 conversations, Revise turns = 14, Revise-containing conversations = 11
Fold 4: 10 conversations, Revise turns = 12, Revise-containing conversations = 10
Fold 5: 10 conversations, Revise turns = 12, Revise-containing conversations = 10


In [25]:
chitchat_df = (
    work_df[
        (work_df["Chitchat"] > 0)
        & (~work_df["conv_id"].isin(assigned_ids))
    ]
    .sort_values(
        ["Chitchat", "shuffle_order"],
        ascending=[False, True],
        kind="stable"
    )
)

print("Remaining Chitchat conversations:", len(chitchat_df))

assign_round_robin(chitchat_df)

Remaining Chitchat conversations: 195


In [26]:
remaining_df = (
    work_df[
        ~work_df["conv_id"].isin(assigned_ids)
    ]
    .sort_values(
        "shuffle_order",
        kind="stable"
    )
)

print("Remaining conversations:", len(remaining_df))

assign_round_robin(remaining_df)

Remaining conversations: 577


In [27]:
for f in range(1, N_FOLDS + 1):
    print(
        f"Fold {f}: {len(folds[f])} conversations"
    )

Fold 1: 165 conversations
Fold 2: 165 conversations
Fold 3: 165 conversations
Fold 4: 165 conversations
Fold 5: 165 conversations


In [28]:
summary_rows = []

for f in range(1, N_FOLDS + 1):

    subset = work_df[
        work_df["conv_id"].isin(folds[f])
    ]

    summary_rows.append({
        "Fold": f,
        "Conversations": len(subset),
        "Reveal": int(subset["Reveal"].sum()),
        "Interpret": int(subset["Interpret"].sum()),
        "Revise": int(subset["Revise"].sum()),
        "Chitchat": int(subset["Chitchat"].sum()),
        "Total_examples": int(subset["total_4class"].sum())
    })

fold_summary_df = pd.DataFrame(summary_rows)

display(fold_summary_df)

,Fold,Conversations,Reveal,Interpret,Revise,Chitchat,Total_examples
0,1,165,210,377,15,53,655
1,2,165,201,358,14,50,623
2,3,165,217,351,14,52,634
3,4,165,201,368,12,53,634
4,5,165,202,359,12,60,633


In [29]:
# ============================================================
# CV1 OUTER SPLIT
# Fold 1 = TEST
# Folds 2, 3, 4, 5 = development pool
# ============================================================

CV_RUN = 1
TEST_FOLD = 1

# Fold 1: held-out test conversation IDs
cv1_test_ids = list(folds[TEST_FOLD])

# Remaining 4 folds: development conversation IDs
cv1_dev_ids = []

for fold_id in range(1, N_FOLDS + 1):
    if fold_id != TEST_FOLD:
        cv1_dev_ids.extend(folds[fold_id])

print("CV1")
print("Test fold:", TEST_FOLD)
print("Test conversations:", len(cv1_test_ids))
print("Development conversations:", len(cv1_dev_ids))
print("Total:", len(cv1_test_ids) + len(cv1_dev_ids))

CV1
Test fold: 1
Test conversations: 165
Development conversations: 660
Total: 825


In [30]:
# Check that test and development conversations do not overlap

test_set = set(cv1_test_ids)
dev_set = set(cv1_dev_ids)

overlap = test_set & dev_set

print("Test unique IDs:", len(test_set))
print("Development unique IDs:", len(dev_set))
print("Overlap:", len(overlap))

assert len(test_set) == 165
assert len(dev_set) == 660
assert len(overlap) == 0

assert len(test_set | dev_set) == N_CONVERSATIONS

print("\nPASS: CV1 test and development sets are conversation-disjoint.")

Test unique IDs: 165
Development unique IDs: 660
Overlap: 0

PASS: CV1 test and development sets are conversation-disjoint.


In [31]:
# Extract complete raw conversations for CV1

cv1_test_raw = [
    conv
    for conv in raw_data
    if conv["conv_id"] in test_set
]

cv1_dev_raw = [
    conv
    for conv in raw_data
    if conv["conv_id"] in dev_set
]

print("CV1 raw test conversations:", len(cv1_test_raw))
print("CV1 raw development conversations:", len(cv1_dev_raw))

CV1 raw test conversations: 165
CV1 raw development conversations: 660


In [32]:
# Inspect one complete conversation from the CV1 test set

print(json.dumps(
    cv1_test_raw[0],
    indent=2,
    ensure_ascii=False
))

{
  "conv_id": 20771,
  "conversation": [
    {
      "msg_id": 12146,
      "turn_id": 1,
      "role": "user",
      "content": "I am looking for a new laptop",
      "intent": "Reveal",
      "keywords": [
        "laptop"
      ]
    },
    {
      "msg_id": 12155,
      "turn_id": 1,
      "role": "system",
      "content": "Hello, which operating system do you prefer?",
      "action": "Clarify",
      "clarifying_attribute": [
        "Operation System"
      ]
    },
    {
      "msg_id": 12164,
      "turn_id": 2,
      "role": "user",
      "content": "I use windows",
      "intent": "Interpret",
      "keywords": [
        "windows laptop"
      ]
    },
    {
      "msg_id": 12176,
      "turn_id": 2,
      "role": "system",
      "content": "Do you have a brand or price range in mind?",
      "action": "Clarify",
      "clarifying_attribute": [
        "Brand",
        "Price: $        $"
      ]
    },
    {
      "msg_id": 12190,
      "turn_id": 2,
      "role": "user",

In [33]:
# Inspect one complete conversation from the CV1 development pool

print(json.dumps(
    cv1_dev_raw[0],
    indent=2,
    ensure_ascii=False
))

{
  "conv_id": 19762,
  "conversation": [
    {
      "msg_id": 11140,
      "turn_id": 1,
      "role": "user",
      "content": "Hello, i would like to purchase a new fridge",
      "intent": "Reveal",
      "keywords": [
        "fridge"
      ]
    },
    {
      "msg_id": 11152,
      "turn_id": 1,
      "role": "system",
      "content": "What size fridge are you looking for?",
      "action": "Clarify",
      "clarifying_attribute": [
        "Size"
      ]
    },
    {
      "msg_id": 11172,
      "turn_id": 1,
      "role": "system",
      "content": "Also in what price range are you looking?",
      "action": "Clarify",
      "clarifying_attribute": [
        "Price: $        $"
      ]
    },
    {
      "msg_id": 11191,
      "turn_id": 1,
      "role": "user",
      "content": "I am looking for a new table sized fridge, since my old one broke.",
      "intent": "Interpret",
      "keywords": [
        "table size fridge"
      ]
    },
    {
      "msg_id": 11219,
      "t

In [34]:
# Class distribution in CV1 TEST

cv1_test_audit = eligible_df[
    eligible_df["conv_id"].isin(test_set)
]

cv1_test_class_counts = (
    cv1_test_audit[TARGET_INTENTS]
    .sum()
)

print("CV1 TEST")
print("Conversations:", len(cv1_test_audit))
print("\nClass counts:")
print(cv1_test_class_counts)

print(
    "\nTotal target user turns:",
    cv1_test_class_counts.sum()
)

CV1 TEST
Conversations: 165

Class counts:
Reveal       210
Interpret    377
Revise        15
Chitchat      53
dtype: int64

Total target user turns: 655


In [35]:
# Class distribution in CV1 development pool

cv1_dev_audit = eligible_df[
    eligible_df["conv_id"].isin(dev_set)
]

cv1_dev_class_counts = (
    cv1_dev_audit[TARGET_INTENTS]
    .sum()
)

print("CV1 DEVELOPMENT POOL")
print("Conversations:", len(cv1_dev_audit))

print("\nClass counts:")
print(cv1_dev_class_counts)

print(
    "\nTotal target user turns:",
    cv1_dev_class_counts.sum()
)

CV1 DEVELOPMENT POOL
Conversations: 660

Class counts:
Reveal        821
Interpret    1436
Revise         52
Chitchat      215
dtype: int64

Total target user turns: 2524


In [36]:
# ============================================================
# CV1: split development pool into TRAIN and VALIDATION
# 80% train / 20% validation
# conversation-level random split
# ============================================================

INNER_SEED = 42

# Work on a copy so cv1_dev_ids itself is not modified
cv1_dev_ids_shuffled = cv1_dev_ids.copy()

# Reproducible shuffle
random.Random(INNER_SEED).shuffle(cv1_dev_ids_shuffled)

N_DEV = len(cv1_dev_ids_shuffled)

N_TRAIN = int(N_DEV * 0.80)
N_VAL = N_DEV - N_TRAIN

cv1_train_ids = cv1_dev_ids_shuffled[:N_TRAIN]
cv1_val_ids = cv1_dev_ids_shuffled[N_TRAIN:]

print("CV1 development:", N_DEV)
print("CV1 train:", len(cv1_train_ids))
print("CV1 validation:", len(cv1_val_ids))
print("CV1 test:", len(cv1_test_ids))

CV1 development: 660
CV1 train: 528
CV1 validation: 132
CV1 test: 165


In [37]:
train_set = set(cv1_train_ids)
val_set = set(cv1_val_ids)
test_set = set(cv1_test_ids)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

assert len(train_set) == 528
assert len(val_set) == 132
assert len(test_set) == 165

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

assert len(train_set | val_set | test_set) == 825

print("PASS: CV1 train/validation/test are conversation-disjoint.")

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS: CV1 train/validation/test are conversation-disjoint.


In [38]:
def split_summary(ids):
    subset = eligible_df[
        eligible_df["conv_id"].isin(ids)
    ]

    counts = subset[TARGET_INTENTS].sum()

    return {
        "Conversations": len(subset),
        "Reveal": int(counts["Reveal"]),
        "Interpret": int(counts["Interpret"]),
        "Revise": int(counts["Revise"]),
        "Chitchat": int(counts["Chitchat"]),
        "Total_examples": int(counts.sum())
    }


cv1_split_summary = pd.DataFrame([
    {"Split": "Train", **split_summary(cv1_train_ids)},
    {"Split": "Validation", **split_summary(cv1_val_ids)},
    {"Split": "Test", **split_summary(cv1_test_ids)}
])

display(cv1_split_summary)

,Split,Conversations,Reveal,Interpret,Revise,Chitchat,Total_examples
0,Train,528,671,1143,45,176,2035
1,Validation,132,150,293,7,39,489
2,Test,165,210,377,15,53,655


In [39]:
import os
import pandas as pd

SPLIT_DIR = "/content/drive/MyDrive/5cv_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

pd.DataFrame({"conv_id": cv1_train_ids}).to_csv(
    f"{SPLIT_DIR}/cv1_train_ids.csv",
    index=False
)

pd.DataFrame({"conv_id": cv1_val_ids}).to_csv(
    f"{SPLIT_DIR}/cv1_val_ids.csv",
    index=False
)

pd.DataFrame({"conv_id": cv1_test_ids}).to_csv(
    f"{SPLIT_DIR}/cv1_test_ids.csv",
    index=False
)

print("CV1 frozen:")
print("Train:", len(cv1_train_ids))
print("Validation:", len(cv1_val_ids))
print("Test:", len(cv1_test_ids))

CV1 frozen:
Train: 528
Validation: 132
Test: 165


In [40]:
# ============================================================
# Helper: build one CV run
# test_fold = 1..5
# outer test = one fold
# remaining 4 folds = 660 dev conversations
# dev -> random 80/20 train/validation split
# ============================================================

def build_cv_split(test_fold, folds, eligible_df, inner_seed=42):

    # -----------------------------
    # 1. TEST IDs
    # -----------------------------
    test_ids = list(folds[test_fold])

    # -----------------------------
    # 2. DEV IDs = remaining 4 folds
    # -----------------------------
    dev_ids = []

    for fold_id in range(1, N_FOLDS + 1):
        if fold_id != test_fold:
            dev_ids.extend(folds[fold_id])

    # -----------------------------
    # 3. Random 80/20 split of DEV
    # -----------------------------
    dev_ids_shuffled = dev_ids.copy()

    random.Random(inner_seed).shuffle(dev_ids_shuffled)

    n_dev = len(dev_ids_shuffled)
    n_train = int(n_dev * 0.80)

    train_ids = dev_ids_shuffled[:n_train]
    val_ids = dev_ids_shuffled[n_train:]

    return {
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids,
        "dev_ids": dev_ids
    }

In [41]:
cv_splits = {}

for cv_run in range(1, N_FOLDS + 1):

    cv_splits[cv_run] = build_cv_split(
        test_fold=cv_run,
        folds=folds,
        eligible_df=eligible_df,
        inner_seed=42
    )

    print(
        f"CV{cv_run}: "
        f"Train={len(cv_splits[cv_run]['train_ids'])}, "
        f"Val={len(cv_splits[cv_run]['val_ids'])}, "
        f"Test={len(cv_splits[cv_run]['test_ids'])}"
    )

CV1: Train=528, Val=132, Test=165
CV2: Train=528, Val=132, Test=165
CV3: Train=528, Val=132, Test=165
CV4: Train=528, Val=132, Test=165
CV5: Train=528, Val=132, Test=165


In [42]:
# ============================================================
# LEAKAGE CHECK FOR ALL 5 CV RUNS
# ============================================================

for cv_run in range(1, N_FOLDS + 1):

    train_ids = set(cv_splits[cv_run]["train_ids"])
    val_ids = set(cv_splits[cv_run]["val_ids"])
    test_ids = set(cv_splits[cv_run]["test_ids"])

    print(f"\n===== CV{cv_run} =====")

    print("Train:", len(train_ids))
    print("Validation:", len(val_ids))
    print("Test:", len(test_ids))

    print("Train ∩ Val:", len(train_ids & val_ids))
    print("Train ∩ Test:", len(train_ids & test_ids))
    print("Val ∩ Test:", len(val_ids & test_ids))

    assert len(train_ids) == 528
    assert len(val_ids) == 132
    assert len(test_ids) == 165

    assert len(train_ids & val_ids) == 0
    assert len(train_ids & test_ids) == 0
    assert len(val_ids & test_ids) == 0

    assert len(train_ids | val_ids | test_ids) == 825

    print("PASS: no conversation leakage.")


===== CV1 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
PASS: no conversation leakage.

===== CV2 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
PASS: no conversation leakage.

===== CV3 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
PASS: no conversation leakage.

===== CV4 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
PASS: no conversation leakage.

===== CV5 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
PASS: no conversation leakage.


In [43]:
def get_class_summary(ids):

    subset = eligible_df[
        eligible_df["conv_id"].isin(ids)
    ]

    counts = subset[TARGET_INTENTS].sum()

    return {
        "Conversations": len(subset),
        "Reveal": int(counts["Reveal"]),
        "Interpret": int(counts["Interpret"]),
        "Revise": int(counts["Revise"]),
        "Chitchat": int(counts["Chitchat"]),
        "Total_examples": int(counts.sum())
    }


all_cv_summary = []

for cv_run in range(1, N_FOLDS + 1):

    split = cv_splits[cv_run]

    all_cv_summary.append({
        "CV": cv_run,
        "Split": "Train",
        **get_class_summary(split["train_ids"])
    })

    all_cv_summary.append({
        "CV": cv_run,
        "Split": "Validation",
        **get_class_summary(split["val_ids"])
    })

    all_cv_summary.append({
        "CV": cv_run,
        "Split": "Test",
        **get_class_summary(split["test_ids"])
    })


all_cv_summary_df = pd.DataFrame(all_cv_summary)

display(all_cv_summary_df)

,CV,Split,Conversations,Reveal,Interpret,Revise,Chitchat,Total_examples
0,1,Train,528,671,1143,45,176,2035
1,1,Validation,132,150,293,7,39,489
2,1,Test,165,210,377,15,53,655
3,2,Train,528,670,1158,46,179,2053
4,2,Validation,132,160,297,7,39,503
5,2,Test,165,201,358,14,50,623
6,3,Train,528,653,1163,46,179,2041
7,3,Validation,132,161,299,7,37,504
8,3,Test,165,217,351,14,52,634
9,4,Train,528,665,1146,48,179,2038


In [44]:
all_test_ids = []

for cv_run in range(1, N_FOLDS + 1):
    all_test_ids.extend(cv_splits[cv_run]["test_ids"])

print("Total test assignments:", len(all_test_ids))
print("Unique test conversations:", len(set(all_test_ids)))

assert len(all_test_ids) == 825
assert len(set(all_test_ids)) == 825

print(
    "PASS: every eligible conversation appears "
    "in the outer test set exactly once."
)

Total test assignments: 825
Unique test conversations: 825
PASS: every eligible conversation appears in the outer test set exactly once.


In [45]:
import os
import pandas as pd

SPLIT_DIR = "/content/drive/MyDrive/5cv_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

for fold_id in range(1, N_FOLDS + 1):
    pd.DataFrame({
        "conv_id": folds[fold_id]
    }).to_csv(
        f"{SPLIT_DIR}/fold_{fold_id}_ids.csv",
        index=False
    )

print("Saved 5 outer folds.")

Saved 5 outer folds.


In [46]:
pd.DataFrame({
    "conv_id": cv1_train_ids
}).to_csv(
    f"{SPLIT_DIR}/cv1_train_ids.csv",
    index=False
)

pd.DataFrame({
    "conv_id": cv1_val_ids
}).to_csv(
    f"{SPLIT_DIR}/cv1_val_ids.csv",
    index=False
)

pd.DataFrame({
    "conv_id": cv1_test_ids
}).to_csv(
    f"{SPLIT_DIR}/cv1_test_ids.csv",
    index=False
)

print("CV1 saved.")
print("Train:", len(cv1_train_ids))
print("Validation:", len(cv1_val_ids))
print("Test:", len(cv1_test_ids))

CV1 saved.
Train: 528
Validation: 132
Test: 165


In [47]:
check_train = pd.read_csv(
    f"{SPLIT_DIR}/cv1_train_ids.csv"
)["conv_id"].tolist()

check_val = pd.read_csv(
    f"{SPLIT_DIR}/cv1_val_ids.csv"
)["conv_id"].tolist()

check_test = pd.read_csv(
    f"{SPLIT_DIR}/cv1_test_ids.csv"
)["conv_id"].tolist()

print("Saved CV1 train:", len(check_train))
print("Saved CV1 val:", len(check_val))
print("Saved CV1 test:", len(check_test))

assert set(check_train) == set(cv1_train_ids)
assert set(check_val) == set(cv1_val_ids)
assert set(check_test) == set(cv1_test_ids)

assert set(check_train).isdisjoint(check_val)
assert set(check_train).isdisjoint(check_test)
assert set(check_val).isdisjoint(check_test)

print("PASS: saved CV1 files match the approved split.")

Saved CV1 train: 528
Saved CV1 val: 132
Saved CV1 test: 165
PASS: saved CV1 files match the approved split.


In [48]:
# ============================================================
# Build CV2, CV3, CV4, CV5
# Same protocol used for CV1
# ============================================================

INNER_SEED = 42

remaining_cv_splits = {}

for cv_run in range(2, N_FOLDS + 1):

    # -----------------------------------
    # TEST = corresponding outer fold
    # -----------------------------------
    test_ids = list(folds[cv_run])

    # -----------------------------------
    # DEV = remaining 4 folds
    # -----------------------------------
    dev_ids = []

    for fold_id in range(1, N_FOLDS + 1):
        if fold_id != cv_run:
            dev_ids.extend(folds[fold_id])

    # -----------------------------------
    # Random 80/20 train-validation split
    # -----------------------------------
    dev_ids_shuffled = dev_ids.copy()

    random.Random(INNER_SEED).shuffle(
        dev_ids_shuffled
    )

    n_dev = len(dev_ids_shuffled)
    n_train = int(n_dev * 0.80)

    train_ids = dev_ids_shuffled[:n_train]
    val_ids = dev_ids_shuffled[n_train:]

    remaining_cv_splits[cv_run] = {
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids,
        "dev_ids": dev_ids
    }

    print(
        f"CV{cv_run}: "
        f"Train={len(train_ids)}, "
        f"Val={len(val_ids)}, "
        f"Test={len(test_ids)}"
    )

CV2: Train=528, Val=132, Test=165
CV3: Train=528, Val=132, Test=165
CV4: Train=528, Val=132, Test=165
CV5: Train=528, Val=132, Test=165


In [49]:
# ============================================================
# Conversation-level leakage check
# ============================================================

for cv_run in range(2, N_FOLDS + 1):

    split = remaining_cv_splits[cv_run]

    train_set = set(split["train_ids"])
    val_set = set(split["val_ids"])
    test_set = set(split["test_ids"])

    print(f"\n===== CV{cv_run} =====")

    print("Train:", len(train_set))
    print("Validation:", len(val_set))
    print("Test:", len(test_set))

    print("Train ∩ Validation:", len(train_set & val_set))
    print("Train ∩ Test:", len(train_set & test_set))
    print("Validation ∩ Test:", len(val_set & test_set))

    # Size checks
    assert len(train_set) == 528
    assert len(val_set) == 132
    assert len(test_set) == 165

    # Leakage checks
    assert len(train_set & val_set) == 0
    assert len(train_set & test_set) == 0
    assert len(val_set & test_set) == 0

    # Must reconstruct all 825 conversations
    assert len(train_set | val_set | test_set) == N_CONVERSATIONS

    print(f"PASS: CV{cv_run} has no conversation leakage.")


===== CV2 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS: CV2 has no conversation leakage.

===== CV3 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS: CV3 has no conversation leakage.

===== CV4 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS: CV4 has no conversation leakage.

===== CV5 =====
Train: 528
Validation: 132
Test: 165
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
PASS: CV5 has no conversation leakage.


In [50]:
def get_split_summary(ids):

    subset = eligible_df[
        eligible_df["conv_id"].isin(ids)
    ]

    counts = subset[TARGET_INTENTS].sum()

    return {
        "Conversations": len(subset),
        "Reveal": int(counts["Reveal"]),
        "Interpret": int(counts["Interpret"]),
        "Revise": int(counts["Revise"]),
        "Chitchat": int(counts["Chitchat"]),
        "Total_examples": int(counts.sum())
    }


rows = []

for cv_run in range(2, N_FOLDS + 1):

    split = remaining_cv_splits[cv_run]

    for split_name, ids_key in [
        ("Train", "train_ids"),
        ("Validation", "val_ids"),
        ("Test", "test_ids")
    ]:

        rows.append({
            "CV": f"CV{cv_run}",
            "Split": split_name,
            **get_split_summary(split[ids_key])
        })

cv2_to_cv5_summary_df = pd.DataFrame(rows)

display(cv2_to_cv5_summary_df)

,CV,Split,Conversations,Reveal,Interpret,Revise,Chitchat,Total_examples
0,CV2,Train,528,670,1158,46,179,2053
1,CV2,Validation,132,160,297,7,39,503
2,CV2,Test,165,201,358,14,50,623
3,CV3,Train,528,653,1163,46,179,2041
4,CV3,Validation,132,161,299,7,37,504
5,CV3,Test,165,217,351,14,52,634
6,CV4,Train,528,665,1146,48,179,2038
7,CV4,Validation,132,165,299,7,36,507
8,CV4,Test,165,201,368,12,53,634
9,CV5,Train,528,664,1136,48,172,2020


In [51]:
# ============================================================
# Freeze CV2-CV5 split IDs
# ============================================================

for cv_run in range(2, N_FOLDS + 1):

    split = remaining_cv_splits[cv_run]

    pd.DataFrame({
        "conv_id": split["train_ids"]
    }).to_csv(
        f"{SPLIT_DIR}/cv{cv_run}_train_ids.csv",
        index=False
    )

    pd.DataFrame({
        "conv_id": split["val_ids"]
    }).to_csv(
        f"{SPLIT_DIR}/cv{cv_run}_val_ids.csv",
        index=False
    )

    pd.DataFrame({
        "conv_id": split["test_ids"]
    }).to_csv(
        f"{SPLIT_DIR}/cv{cv_run}_test_ids.csv",
        index=False
    )

    print(f"CV{cv_run} saved.")

CV2 saved.
CV3 saved.
CV4 saved.
CV5 saved.


In [52]:
all_test_ids = []

# CV1
all_test_ids.extend(cv1_test_ids)

# CV2-CV5
for cv_run in range(2, N_FOLDS + 1):
    all_test_ids.extend(
        remaining_cv_splits[cv_run]["test_ids"]
    )

print("Total test assignments:", len(all_test_ids))
print("Unique test conversations:", len(set(all_test_ids)))

assert len(all_test_ids) == N_CONVERSATIONS
assert len(set(all_test_ids)) == N_CONVERSATIONS

print(
    "PASS: every eligible conversation is used "
    "as outer test data exactly once."
)

Total test assignments: 825
Unique test conversations: 825
PASS: every eligible conversation is used as outer test data exactly once.
